In [1]:
import os
from pyspark import SparkConf
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

# Adding AWS S3 Minio configs
sparkConf = (
    SparkConf()
    .set("spark.jars.ivy","/home/brijeshdhaker/.ivy2")
    #.set("spark.jars.packages","org.apache.hadoop:hadoop-aws:3.0.0")
    #.set("spark.executor.heartbeatInterval", "300000")
    #.set("spark.network.timeout", "400000")
    #.set("spark.hadoop.fs.s3a.endpoint", "http://minio.sandbox.net:9010")
    #.set("spark.hadoop.fs.s3a.access.key", "pgm2H2bR7a5kMc5XCYdO")
    #.set("spark.hadoop.fs.s3a.secret.key", "zjd8T0hXFGtfemVQ6AH3yBAPASJNXNbVSx5iddqG")
    #.set("spark.hadoop.fs.s3a.path.style.access", "true")
    #.set("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider")
    #.set("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    #.set("spark.sql.warehouse.dir", "s3a://defaultfs/spark/warehouse")
    #.set("spark.hadoop.fs.defaultFS", "s3a://defaultfs/")
    #.set("spark.eventLog.enabled", "true")
    #.set("spark.eventLog.dir", "file:///apps/var/logs/spark-events")
)

spark = (
    SparkSession.builder.master("local[*]").
        appName('spark-null-handling').
        config(conf=sparkConf).
        getOrCreate()
)

spark.sparkContext.setLogLevel('ERROR')
spark

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/02/05 09:32:27 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
payment_col= ["paymentId", "customerId","amount"]

payment_data = [
    (1, 101, 2500), 
    (2, 102, 1110), 
    (3, 103, 500), 
    (4, 104, 400), 
    (5, 105, 150), 
    (6, 106, 450)
]

payment_df = spark.createDataFrame(payment_data,payment_col)
payment_df.show()

customer_col = ["customerId", "name"]

customer_data = [
    (101,"Jon"), 
    (102,"Aron"),
    (103,"Sam")
]

customer_df = spark.createDataFrame(customer_data,customer_col)
customer_df.show()

+---------+----------+------+
|paymentId|customerId|amount|
+---------+----------+------+
|        1|       101|  2500|
|        2|       102|  1110|
|        3|       103|   500|
|        4|       104|   400|
|        5|       105|   150|
|        6|       106|   450|
+---------+----------+------+

+----------+----+
|customerId|name|
+----------+----+
|       101| Jon|
|       102|Aron|
|       103| Sam|
+----------+----+



In [3]:
leftJoinDf = payment_df.join(customer_df, customer_df["customerId"] == payment_df["customerId"], "left" )
leftJoinDf.show()

+---------+----------+------+----------+----+
|paymentId|customerId|amount|customerId|name|
+---------+----------+------+----------+----+
|        1|       101|  2500|       101| Jon|
|        2|       102|  1110|       102|Aron|
|        3|       103|   500|       103| Sam|
|        4|       104|   400|      NULL|NULL|
|        5|       105|   150|      NULL|NULL|
|        6|       106|   450|      NULL|NULL|
+---------+----------+------+----------+----+



In [4]:
from pyspark.sql.functions import asc_nulls_last

# Sort "department" ascending, with nulls at the end
leftJoinDf.sort(asc_nulls_last("name")).show()

+---------+----------+------+----------+----+
|paymentId|customerId|amount|customerId|name|
+---------+----------+------+----------+----+
|        2|       102|  1110|       102|Aron|
|        1|       101|  2500|       101| Jon|
|        3|       103|   500|       103| Sam|
|        4|       104|   400|      NULL|NULL|
|        5|       105|   150|      NULL|NULL|
|        6|       106|   450|      NULL|NULL|
+---------+----------+------+----------+----+



In [11]:
#
## Replacing null values with some value
#
leftJoinDf.fillna(value="").show()

#
#leftJoinDf.fillna(value=0,subset=["customerId"]).show() # only for customerId column

#
# leftJoinDf.fillna(10,["customerId"]) \
    #.fillna("---",["name"]).show()

+---------+----------+------+----------+----+
|paymentId|customerId|amount|customerId|name|
+---------+----------+------+----------+----+
|        1|       101|  2500|       101| Jon|
|        2|       102|  1110|       102|Aron|
|        3|       103|   500|       103| Sam|
|        4|       104|   400|      NULL|    |
|        5|       105|   150|      NULL|    |
|        6|       106|   450|      NULL|    |
+---------+----------+------+----------+----+



In [ ]:
#
## Replacing null values with some value using na
#
leftJoinDf.na.fill(value=0).show()
leftJoinDf.na.fill(value=0,subset=["emp_comm"]).show()

In [ ]:
# Drop rows with nulls in any column
df_dropped_any = leftJoinDf.dropna()

# Drop rows only if ALL columns are null
df_dropped_all = leftJoinDf.dropna(how="all")

# Drop rows with nulls only in the "Name" or "Score" columns
df_dropped_subset = leftJoinDf.dropna(subset=["customerId", "name"])


In [5]:
from pyspark.sql.functions import col

# Keep rows where the "Name" column is not null
df_filtered_col = leftJoinDf.filter(col("name").isNotNull())
df_filtered_col.show()

# Alternatively, using dot notation (if column name has no spaces)
df_filtered_dot = leftJoinDf.filter(leftJoinDf.name.isNotNull())
df_filtered_dot.show()

+---------+----------+------+----------+----+
|paymentId|customerId|amount|customerId|name|
+---------+----------+------+----------+----+
|        1|       101|  2500|       101| Jon|
|        2|       102|  1110|       102|Aron|
|        3|       103|   500|       103| Sam|
+---------+----------+------+----------+----+

+---------+----------+------+----------+----+
|paymentId|customerId|amount|customerId|name|
+---------+----------+------+----------+----+
|        1|       101|  2500|       101| Jon|
|        2|       102|  1110|       102|Aron|
|        3|       103|   500|       103| Sam|
+---------+----------+------+----------+----+



In [7]:
# Filter using SQL expression
df_filtered_sql = leftJoinDf.filter("name IS NOT NULL")
df_filtered_sql.show()

# To also filter out empty strings, you can combine conditions customerId IS NOT NULL AND name != ''
df_filtered_sql_clean = leftJoinDf.filter("name != ''")
df_filtered_sql_clean.show()

+---------+----------+------+----------+----+
|paymentId|customerId|amount|customerId|name|
+---------+----------+------+----------+----+
|        1|       101|  2500|       101| Jon|
|        2|       102|  1110|       102|Aron|
|        3|       103|   500|       103| Sam|
+---------+----------+------+----------+----+

+---------+----------+------+----------+----+
|paymentId|customerId|amount|customerId|name|
+---------+----------+------+----------+----+
|        1|       101|  2500|       101| Jon|
|        2|       102|  1110|       102|Aron|
|        3|       103|   500|       103| Sam|
+---------+----------+------+----------+----+

